# Feature mathematics and scenario proxies

- Explains the reusable calculations behind possession analysis.
- Defines a scenario as a measurable pattern over frames, not a hand-written label.
- Treats every proxy as evidence with explicit limitations.

- ATD provides normalised 2D ball and temporary-slot coordinates.
- A slot is a coordinate stream, not a guaranteed persistent player identity.
- `ball_z` is unavailable: fast 2D movement is measurable, aerial height is not.
- Smart Tagging is context, not a frame-level possession label.

## 1. Coordinate system and distance

- Convert normalised ATD coordinates to metres using `105.3 m × 68.0 m`.
- If the ball is `(b_x, b_y)` and a slot is `(p_x, p_y)`, use:

`d = sqrt(((p_x - b_x) * L)^2 + ((p_y - b_y) * W)^2)`

- Calculate this for every visible slot on every frame.
- `closest_team` is the team of the smallest-distance slot; `closest_distance_m` is that distance.
- Use it as a comfortable-control anchor only; passes, races, contests, touches, and slot swaps can mislead.

## 2. Team margin, zones, and pressure

- Let `d_H` and `d_A` be nearest Home/Away distances.
- Signed margin: `d_A - d_H`; positive favours Home, negative favours Away.
- Absolute margin measures how decisive the race is.

- Radius counts: visible H/A slots with `d <= r`.
- Counts describe support/pressure, not control by themselves.
- High opponent density makes nearest-player possession unreliable.
- Possession-zone proxy = small distance + meaningful margin + low opponent density.
- Thresholds come from configuration, not copied labels.

## 3. Time-series features

- Lagged change: `d_t - d_(t-k)`.
- Closing rate: `(d_(t-k) - d_t) / (k / FPS)`; positive means approaching.
- Rolling mean/minimum/std describe recent level and volatility.
- Persistence is the fraction of a trailing window agreeing with H or A.

- `Δx = (x_t-x_(t-1))*L`; `Δy = (y_t-y_(t-1))*W`.
- Displacement: `sqrt(Δx²+Δy²)`; speed: `displacement*FPS`.
- Direction: `atan2(Δy, Δx)`; acceleration: speed change × FPS.
- A speed drop supports reception evidence but does not prove a touch.

## 4. Missing-ball trajectory proxy

For an internal missing interval, the last observed point before the gap and the first observed point after it form a bridge. We calculate gap duration, displacement, implied speed, direction change, and the residual from a straight-line path. We also inspect nearby coordinate streams: a player whose position closes toward the bridge can support a reception hypothesis.

This is deliberately not an automatic possession fill. A short, coherent bridge between same-team anchors can be filled by a configured rule; a long or physically implausible bridge remains unresolved or `X`. The plot uses 75 frames before and after for human inspection, while the resolver uses its smaller configured trajectory context.

## 5. Reachability and reception evidence

For each team, the reachability layer estimates the fastest arrival to the current or projected ball location from player distance and configured speed assumptions. Useful reception evidence includes: a local minimum in ball-to-player distance, decreasing receiver distance, ball deceleration, agreement between nearest-team and projected receiver team, persistence after arrival, and teammate/opponent proximity.

A genuine reception should be supported after the arrival: the candidate receiver remains close for a persistence window or the team establishes a comfortable-control anchor. A one-frame touch has proximity without persistence. A switch requires a receiver from the other team plus sustained post-arrival evidence; merely touching or being closest does not force a switch.

## 6. Ball-in-flight and scenario cases

`BALL_IN_FLIGHT` is a derived 2D proxy: recent ball speed reaches the configured minimum (currently 8 m/s) and the closest player is outside the stable-control radius (currently 2 m). The decoder also excludes dead-ball/restart, reception, tracking-artifact, and miscontrol candidates. `RECEPTION_CANDIDATE` is a flight episode where arrival-time evidence is close between teams. No `z` coordinate means this includes fast ground passes and through-balls; it is not an aerial-ball detector.

The operational cases are:

- **Comfortable control:** small distance, decisive team margin, low opponent pressure, and persistence.
- **Contested ball:** both teams close or local opponent density is high; nearest-player output is treated as ambiguous.
- **Ball in flight:** fast live-ball movement without a stable controller. Keep the prior team until reception evidence confirms a change.
- **Reception candidate:** projected receiver arrival is competitive; inspect the post-arrival persistence.
- **Miscontrol / one-frame touch:** close approach followed by rapid separation without sustained control.
- **Tracking artifact / missing ball:** coordinates are absent or physically incoherent; use bounded before/after evidence, not a global fill.
- **Restart / dead ball:** Smart Tagging set-piece context, out-of-bounds geometry, or sustained whole-pitch low motion. Keep `D` through the restart regime until controlled live possession is evidenced.
- **Confirmed transition:** a later opposing-team reception anchor follows a live ambiguous interval. The interval is assigned according to the configured team-level possession convention.

## 7. Anchors, confidence, and hierarchy

Anchors are frames where evidence is strong enough to constrain the timeline. The hierarchy is configured in `config/possession_rules.yaml`: restart-control anchors first, then pressure anchors, then general control anchors. Scenario evidence without a team is diagnostic only.

Confidence is not a learned probability here. It is a transparent score assembled from evidence such as distance margin, persistence, pressure, reception signal, and restart certainty. A high score can create an anchor; a low score leaves the frame unresolved. Between two anchors, short same-team gaps can be bridged; different-team anchors require reception/transition evidence.

## 8. Modular implementation

The notebook is orchestration; calculations live in reusable functions. `build_engineered_features()` composes the frame table. `add_temporal_features()` adds lagged/rolling movement, ball kinematics, and flight evidence. The reachability, missing-ball, reception, anchor, scenario, and decoder functions add their own columns without using public labels for fitting. Models consume this common table, so a scenario or anchor family can be enabled/disabled by configuration.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'possession' else Path.cwd()
RULES_PATH = PROJECT_ROOT / 'config' / 'possession_rules.yaml'
print('project:', PROJECT_ROOT)
print('rules:', RULES_PATH)

In [ ]:
# Optional executable demonstration. Set True when running against the local data.
RUN_FEATURE_BUILD = False
if RUN_FEATURE_BUILD:
    from possession.calculations.features import build_engineered_features
    from possession.temporal_rules import load_rules
    features = build_engineered_features(rules=load_rules(RULES_PATH))
    print(features.shape)
    display(features[['frame', 'closest_distance_m', 'team_distance_margin_m', 'ball_speed_derived_mps', 'scenario_context']].head())

## Takeaway

The system does not pretend that one nearest player solves possession. It first measures geometry and motion, then identifies the scenario in which that geometry is reliable or unreliable. Strong anchors constrain the timeline; scenario evidence explains ambiguous intervals; bounded bidirectional filling and dynamic-programming decoders enforce temporal consistency.

## 9. Visual evidence: observation → pattern → feature → scenario

The feature table is easiest to audit when the ball, its kinematics, and player geometry are shown together. These plots are exploratory evidence, not hand-written labels.

| Observation | Pattern to recognise | Feature to look for | Scenario supported |
|---|---|---|---|
| Ball moves quickly while no player is in a matching control zone | coherent flight, then a landing/reception region | speed, acceleration, direction, travelled distance, arrival-time race | `BALL_IN_FLIGHT`, `CONTESTED`, long-ball candidate |
| Ball speed drops sharply near an intersecting player path | likely reception/control | deceleration, local distance minimum, path intersection, post-touch persistence | `RECEPTION_CANDIDATE` or controlled possession |
| Ball acceleration rises sharply | departure, pass, deflection, or miscontrol | positive acceleration, direction change, nearest player at departure | kick/pass candidate; do not switch possession yet |
| Ball direction changes after flight | first-time pass, deflection, tackle, or miscontrol | angle change, target-team alignment, player density and approach speed | contested/transition candidate |
| Ball coordinates disappear | observation uncertainty, not zero speed | gap duration, endpoint bridge, implied distance, nearby player paths | missing-ball / unresolved until evidence converges |
| Restart context plus near-zero ball motion | dead-ball setup | Smart Tagging context, pitch boundary, zero/near-zero speed, restart geometry | `D` until live control evidence is visible |

The following cells are guarded: enable `RUN_EVIDENCE_PLOTS` only when the local data files are available.

In [ ]:
# Visual audit block: ball kinematics, missingness, long-ball candidate, and player geometry.
# This cell never creates a label. It ranks an evidence window for inspection.
RUN_EVIDENCE_PLOTS = False

if not RUN_EVIDENCE_PLOTS:
    print("Set RUN_EVIDENCE_PLOTS = True after the local data paths are configured.")
elif 'features' not in globals():
    print("Build `features` first (set RUN_FEATURE_BUILD = True in the cell above).")
else:
    import numpy as np
    import matplotlib.pyplot as plt
    from possession.calculations.plotting import (
        PITCH_LENGTH_M, PITCH_WIDTH_M, plot_trajectory_window,
        select_representative_ball_gap, select_ball_gap_event_window,
    )
    from possession.baseline_pipeline import slot_team_map
    from games import asset_paths

    plot_df = features.copy().sort_values('frame').reset_index(drop=True)
    dx = plot_df.ball_x.diff() * PITCH_LENGTH_M
    dy = plot_df.ball_y.diff() * PITCH_WIDTH_M
    plot_df['plot_step_distance_m'] = np.hypot(dx, dy)
    plot_df['plot_speed_mps'] = plot_df['plot_step_distance_m'] * 25
    plot_df['plot_acceleration_mps2'] = plot_df['plot_speed_mps'].diff() * 25
    plot_df['plot_direction_rad'] = np.arctan2(dy, dx)
    plot_df['plot_direction_change_rad'] = np.arctan2(
        np.sin(plot_df['plot_direction_rad'].diff()),
        np.cos(plot_df['plot_direction_rad'].diff()),
    ).abs()
    plot_df['plot_observed_distance_m'] = plot_df['plot_step_distance_m'].where(
        plot_df.ball_x.notna() & plot_df.ball_y.notna(), 0
    ).fillna(0)

    # 1) Whole-window ball evidence: speed, acceleration, direction change, distance per frame.
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    plot_df.plot(x='frame', y='plot_speed_mps', ax=axes[0], color='#1f77b4', legend=False)
    plot_df.plot(x='frame', y='plot_acceleration_mps2', ax=axes[1], color='#7b3294', legend=False)
    plot_df.plot(x='frame', y='plot_direction_change_rad', ax=axes[2], color='#d95f02', legend=False)
    plot_df.plot(x='frame', y='plot_step_distance_m', ax=axes[3], color='#111111', legend=False)
    axes[0].set_title('Ball evidence across the available window')
    axes[0].set_ylabel('speed (m/s)'); axes[1].set_ylabel('accel. (m/s²)')
    axes[2].set_ylabel('direction change (rad)'); axes[3].set_ylabel('distance/frame (m)')
    axes[1].axhline(0, color='black', linewidth=.7)
    missing = plot_df.ball_x.isna() | plot_df.ball_y.isna()
    if missing.any():
        starts = plot_df.loc[missing & ~missing.shift(fill_value=False), 'frame']
        ends = plot_df.loc[missing & ~missing.shift(-1, fill_value=False), 'frame']
        for left, right in zip(starts, ends):
            for ax in axes: ax.axvspan(left, right, color='#d95f02', alpha=.12)
    plt.tight_layout(); plt.show()

    # 2) Missing-ball audit: show the longest internal gap and its surrounding trajectory.
    gap = select_representative_ball_gap(plot_df)
    if gap is None:
        print('No internal missing-ball interval found in this window.')
    else:
        gap_start, gap_end = gap
        event = select_ball_gap_event_window(plot_df, gap, minimum_context_frames=200, control_radius_m=2.5)
        start, end = event['start_frame'], event['end_frame']
        local = plot_df.loc[plot_df.frame.between(start, end)].copy()
        fig, (ax_speed, ax_accel) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
        local.plot(x='frame', y='plot_speed_mps', ax=ax_speed, color='#1f77b4', legend=False)
        local.plot(x='frame', y='plot_acceleration_mps2', ax=ax_accel, color='#7b3294', legend=False)
        for ax in (ax_speed, ax_accel):
            ax.axvspan(gap_start, gap_end, color='#d95f02', alpha=.2, label='ball coordinates missing')
            ax.legend(loc='upper left')
        ax_speed.set_title(f'Missing-ball audit — observed speed around gap {gap_start}–{gap_end}')
        ax_accel.set_ylabel('acceleration (m/s²)'); plt.tight_layout(); plt.show()
        try:
            team_by_slot = slot_team_map(asset_paths()['atd_meta'])
            fig, ax = plt.subplots(figsize=(11, 7))
            plot_trajectory_window(
                plot_df, start, end, team_by_slot=team_by_slot,
                reference_frame=(gap_start + gap_end)//2, nearby_radius_m=12.0,
                nearby_count=None, zoom_to_involved=True,
                show_all_reference_players=False, touch_radius_m=1.5, ax=ax,
            )
            ax.set_title(f'Missing-ball trajectory — bridge is evidence only; gap {gap_start}–{gap_end}')
            plt.show()
        except Exception as exc:
            print(f'Trajectory plot unavailable: {exc}')

    # 3) Long-ball inspection: rank a steep departure impulse, then measure distance
    # until the first sharp deceleration, direction change, or second acceleration.
    observed = plot_df.dropna(subset=['plot_speed_mps', 'plot_acceleration_mps2']).copy()
    if observed.empty:
        print('No observed ball kinematics available for long-ball inspection.')
    else:
        accel_cut = observed.plot_acceleration_mps2.quantile(.995)
        speed_cut = observed.plot_speed_mps.quantile(.95)
        candidates = observed.loc[
            observed.plot_acceleration_mps2.ge(accel_cut) & observed.plot_speed_mps.ge(speed_cut)
        ].copy()
        if candidates.empty:
            candidates = observed.nlargest(1, 'plot_acceleration_mps2')
        departure = candidates.sort_values(['plot_acceleration_mps2', 'plot_speed_mps'], ascending=False).iloc[0]
        dep_frame = int(departure.frame)
        post = plot_df.loc[plot_df.frame.gt(dep_frame)].copy()
        decel_cut = post.plot_acceleration_mps2.quantile(.05) if not post.empty else np.nan
        turn_cut = plot_df.plot_direction_change_rad.quantile(.99)
        end_candidates = post.loc[
            post.plot_acceleration_mps2.le(decel_cut) |
            post.plot_direction_change_rad.ge(turn_cut) |
            post.plot_acceleration_mps2.ge(accel_cut)
        ] if not post.empty else post
        end_frame = int(end_candidates.iloc[0].frame) if not end_candidates.empty else int(post.frame.iloc[-1]) if not post.empty else dep_frame
        episode = plot_df.loc[plot_df.frame.between(dep_frame, end_frame)].copy()
        episode['distance_from_departure_m'] = episode.plot_step_distance_m.fillna(0).cumsum()
        fig, (ax_speed, ax_dist) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
        ax_speed.plot(episode.frame, episode.plot_speed_mps, label='ball speed', color='#1f77b4')
        ax_speed.plot(episode.frame, episode.plot_acceleration_mps2, label='acceleration', color='#7b3294', alpha=.7)
        ax_dist.plot(episode.frame, episode.distance_from_departure_m, color='#111111', label='distance from departure')
        for ax in (ax_speed, ax_dist):
            ax.axvline(dep_frame, color='#d95f02', linestyle='--', label='departure impulse')
            ax.axvline(end_frame, color='#7f3c8d', linestyle=':', label='next sharp change / touch proxy')
            ax.legend(loc='upper left')
        ax_speed.set_title(f'Long-ball candidate — departure {dep_frame}; episode end {end_frame}')
        ax_speed.set_ylabel('m/s or m/s²'); ax_dist.set_ylabel('metres')
        plt.tight_layout(); plt.show()
        display(pd.DataFrame([{
            'departure_frame': dep_frame,
            'departure_speed_mps': float(departure.plot_speed_mps),
            'departure_acceleration_mps2': float(departure.plot_acceleration_mps2),
            'episode_end_frame': end_frame,
            'distance_from_departure_m': float(episode.distance_from_departure_m.iloc[-1]),
            'interpretation': 'inspect as long-ball/pass/deflection candidate; not an automatic possession label',
        }]))
        try:
            team_by_slot = slot_team_map(asset_paths()['atd_meta'])
            fig, ax = plt.subplots(figsize=(11, 7))
            plot_trajectory_window(
                plot_df, max(int(plot_df.frame.min()), dep_frame - 75), min(int(plot_df.frame.max()), end_frame + 75),
                team_by_slot=team_by_slot, reference_frame=dep_frame,
                nearby_radius_m=12.0, nearby_count=None, zoom_to_involved=True,
                show_all_reference_players=False, touch_radius_m=1.5, ax=ax,
            )
            ax.set_title('Long-ball candidate — ball path and nearby player paths')
            plt.show()
        except Exception as exc:
            print(f'Trajectory plot unavailable: {exc}')
